# 图的构建与输入 — AscendIR 构图与 Parser 解析

在上一章我们认识了 AscendIR 的基本概念。要把一张图交给 GE 编译，第一步是先有一张图——本节聚焦"图从哪里来"。GE 接受两类图输入：**用接口从零手工构图**（C++ 算子原型风格 / ES 极简风格），或**用 Parser 把已有框架模型解析成 Graph**。无论哪条入口，产出的都是 AscendIR `Graph` 对象，均可进入统一的 GE 编译流程。

本节学习大纲如下：

- 两条构图入口总览
- C++ 算子原型构图：什么是算子原型
- C++ 算子原型构图：定义算子与连边
- 各类算子的表达（Data / Const / 必选输入 / 动态输入）
- 设置 Graph 输入输出
- ES 极简构图（C 风格 / C++ / Python）
- Parser 解析框架模型（ONNX / TensorFlow / Caffe）
- Parser 解析配置参数
- 输入规格与构图期校验
- 两种方式的选择建议

> 贯穿本节的最小样例：一张"两个输入相加"的 **Add 图**。算子原型和 ES 都可以从零表达这张图；已有的等价 ONNX 模型也可以通过 Parser 转换成 AscendIR Graph。

## 1. 两条构图入口总览

GE 把"怎么得到一张 AscendIR 图"收敛为两条入口——**全新构建**和 **Parser 解析**。前者从零逐算子搭图，后者复用已有框架模型；二者产出的都是 GE 的 `Graph` 对象，均可进入统一的 GE 编译流程。

<p align="left"><img src="./images/graph_build_overview.svg" alt="两条构图入口总览" width="85%"></p>

| 入口 | 子方式 | 典型接口 | 一句话 |
| --- | --- | --- | --- |
| ① 全新构建 | C++ 算子原型（`op::`） | `op::Data` / `set_input_*` / `Graph::SetInputs` | 用算子原型逐个实例化、连边、组图 |
| ① 全新构建 | ES 极简构图 | `GraphBuilder` / `create_input` / `build_and_reset` | 函数风格，代码量更少 |
| ② Parser 解析 | 解析框架模型 | `aclgrphParseONNX` / `aclgrphParseTensorFlow` / `aclgrphParseCaffe` | 把 onnx/pb 等解析成 Graph |

> 说明：全新构建适合"网络结构清楚、要精细控制"的场景；Parser 解析适合"已有训练好的框架模型，想直接迁移上昇腾"的场景。本节会依次讲清两条入口的写法。

## 2. C++ 算子原型构图：什么是算子原型

用图引擎 C++ 接口全新构图前，必须先理解**算子原型（Operator Prototype）**。算子原型描述了一个算子的**输入、输出、属性**信息（以及它在 AI 处理器上的实现信息）。

算子原型在算子开发阶段通过 `REG_OP` 宏注册，用 `.` 链接 `INPUT` / `OUTPUT` / `ATTR` 等接口，以 `OP_END_FACTORY_REG` 收尾。例如 `SoftmaxV2` 的原型：

```cpp
REG_OP(SoftmaxV2)
    .INPUT(x, TensorType({DT_DOUBLE, DT_FLOAT16, DT_BF16, DT_FLOAT}))
    .OUTPUT(y, TensorType({DT_DOUBLE, DT_FLOAT16, DT_BF16, DT_FLOAT}))
    .ATTR(axes, ListInt, {-1})
    .ATTR(half_to_float, Bool, false)
    .OP_END_FACTORY_REG(SoftmaxV2)
```

注册宏含义如下：

| 宏接口 | 含义 |
| --- | --- |
| `REG_OP(OpType)` | 注册算子类型名 |
| `INPUT(x, TensorType({...}))` | 注册**必选输入** x，及其支持的数据类型列表 |
| `OPTIONAL_INPUT(b, ...)` | 注册**可选输入** b |
| `DYNAMIC_INPUT(a, ...)` | 注册**动态多输入** a（构图时按指定数量实例化 a0、a1… 等输入槽位） |
| `OUTPUT(y, ...)` | 注册输出 y |
| `DYNAMIC_OUTPUT(c, ...)` | 注册动态多输出 |
| `ATTR(x, Type, Default)` | 注册**可选属性**（不设置则用默认值） |
| `REQUIRED_ATTR(d, Type)` | 注册**必选属性**（必须设置，无默认值） |

### 如何获取算子原型

- **内置算子**：从 `${INSTALL_DIR}/opp/built-in/op_graph/inc` 的原型头文件中查看，或查阅"Ascend IR 算子规格说明"。
- **自定义算子**：从 `${INSTALL_DIR}/opp/vendors/<vendor_name>/op_proto/inc` 的自定义算子原型头文件中查看。

其中 `${INSTALL_DIR}` 为 CANN 安装路径，root 用户默认 `/usr/local/Ascend/cann`。

> 注意：构图时你不需要重新写 `REG_OP`，那是算子开发阶段的事。构图阶段只需"读懂"原型——知道某算子有哪些输入、属性，才能正确地实例化和连边。

## 3. C++ 算子原型构图：定义算子与连边

`REG_OP` 注册成功后，会自动生成对应的**衍生接口**：一个 `op::OpType` 类，以及 `set_input_*`、`set_attr_*` 等方法。构图就是"实例化算子 → 设置输入/属性 → 连边"。

### 3.1 包含头文件

```cpp
#include "all_ops.h"   // 内置算子类型，路径 ${INSTALL_DIR}/opp/built-in/op_graph/inc/all_ops.h
#include "graph.h"     // Graph 对象
```

自定义算子则需 include 其原型定义头文件。

### 3.2 创建算子实例

算子可以显式指定名称，也可以省略名称并由 GE 自动生成 `OpType_<序号>`。**同一张图中最终使用的算子名称必须唯一**；为了让图结构更容易识别，通常建议显式命名：

```cpp
auto softmax = op::SoftmaxV2("softmax");
```

### 3.3 设置输入

| 输入类型 | 接口 | 示例 |
| --- | --- | --- |
| 必选 / 可选输入 | `set_input_<输入名>` | `.set_input_x(bias_add_3)` |
| 动态输入 | `create_dynamic_input_<名>` + `set_dynamic_input_<名>` | 见第 4 节 AddN |

### 3.4 设置属性

必选属性、可选属性都用 `set_attr_<属性名>`：

```cpp
auto maxpool1 = op::MaxPool("MaxPool1")
    .set_input_x(tanh1)
    .set_attr_ksize({1, 1, 2, 1})
    .set_attr_strides({1, 1, 2, 1})
    .set_attr_padding("SAME");
```

### 3.5 数据边与控制边

**数据边**：用 `set_input_*` 表达。前一算子只有一个输出时直接传算子；有多个输出时，可以按输出名或输出索引连接。下例假设 `data` 是浮点 Data：

```cpp
auto unique = op::Unique("unique").set_input_x(data);

// 按输出名连接 Unique.y。
auto softplus = op::Softplus("softplus")
                    .set_input_x_by_name(unique, "y");

// 按输出索引连接 Unique.idx；idx 是整数，因此用支持整数输入的 Identity 消费。
auto idx_copy = op::Identity("idx_copy").set_input_x(unique, 1U);

// Sqrt 需要浮点输入，因此连接 Unique.y，而不是整数输出 idx。
auto sqrt = op::Sqrt("sqrt")
                .set_input_x_by_name(unique, "y");
```

**控制边**：仅约束执行顺序、不传数据，用 `AddControlInput` 表达。下例在上面的 `softplus` 上增加控制输入，强制 sqrt 先于 softplus 执行：

```cpp
softplus.AddControlInput(sqrt);
```

## 4. 各类算子的表达（Data / Const / 必选输入 / 动态输入）

不同类别的算子在构图时写法略有差异，下面给出最常用的四类。

### 4.1 Data（数据节点 = 图的输入）

图的输入节点用 `op::Data` 表示。**必须**用 `update_input_desc_x` / `update_output_desc_y` 设置 shape、format、dtype：

```cpp
auto shape_data = std::vector<int64_t>({1, 17, 2, 2});
TensorDesc desc_data(ge::Shape(shape_data), FORMAT_ND, DT_FLOAT);
auto data = op::Data("data");
data.update_input_desc_x(desc_data);     // 必须设置输入描述
data.update_output_desc_y(desc_data);    // 必须设置输出描述
```

### 4.2 Const（常量节点 = 权重 / 偏置）

权重、偏置等常量用 `op::Const`，通过属性 `value` 携带一个 `Tensor`：

```cpp
TensorDesc weight_desc(ge::Shape({1, 3, 3, 3}), FORMAT_NCHW, DT_INT8);
// ... 构造 weighttensor1 数据 ...
Tensor weighttensor1(weight_desc, (uint8_t*)bs_inputData_weight, bs_size_weight * sizeof(int8_t));
auto weight1 = op::Const("Conv2D/weight").set_attr_value(weighttensor1);
```

也可以从 bin 文件读入权重数据（用 `Tensor::SetData` 填入），适合权重较大的真实网络。

### 4.3 必选输入算子

直接用 `set_input_<输入名>` 连接前驱算子即可，如 `SoftmaxV2`：

```cpp
auto softmax = op::SoftmaxV2("Softmax").set_input_x(bias_add_3);
```

### 4.4 动态多输入算子（AddN）

`AddN` 的输入个数不固定，用 `create_dynamic_input_x(n)` 创建 n 个动态输入、`set_dynamic_input_x(i, src)` 逐个设置。Data 节点仍需设置输入、输出 TensorDesc：

```cpp
TensorDesc desc(Shape({2, 3}), FORMAT_ND, DT_FLOAT);
auto data = op::Data("data").set_attr_index(0);
data.update_input_desc_x(desc);
data.update_output_desc_y(desc);

auto addn = op::AddN("addn")
    .create_dynamic_input_x(2)     // 创建 2 个动态输入
    .set_dynamic_input_x(0, data)  // 第 1 个输入
    .set_dynamic_input_x(1, data)  // 第 2 个输入
    .set_attr_N(2);                // 属性 N = 输入个数
```

把同一个 Data 接到两个动态输入是合法的，此时 `AddN` 的语义是 `data + data`。省略算子名称也合法，GE 会自动生成唯一名称；这里显式命名只是为了提高图结构可读性。

> 类似地，动态多输出（如 Split）用 `create_dynamic_output_*`；数据类型转换用 `op::Cast`。完整样例参见 ge-document"Graph 中各类算子表达样例"。

## 5. 设置 Graph 输入输出

算子定义完成后，创建 `Graph` 实例，并用 `SetInputs` / `SetOutputs` 指定图的输入算子和输出算子。下面用本节贯穿的 **Add 图**完整示范"算子原型"风格：

```cpp
#include "all_ops.h"
#include "graph.h"
using namespace ge;

bool MakeAddGraph(Graph &graph) {
  auto shape = std::vector<int64_t>({2, 3});
  TensorDesc desc(ge::Shape(shape), FORMAT_ND, DT_FLOAT);

  // 两个输入：data0(index=0)、data1(index=1)
  auto data0 = op::Data("data0").set_attr_index(0);
  data0.update_input_desc_x(desc);
  data0.update_output_desc_y(desc);
  auto data1 = op::Data("data1").set_attr_index(1);
  data1.update_input_desc_x(desc);
  data1.update_output_desc_y(desc);

  // Add 算子，连接两个 Data
  auto add = op::Add("add").set_input_x1(data0).set_input_x2(data1);

  // 设置图的输入、输出
  std::vector<Operator> inputs{data0, data1};
  std::vector<Operator> outputs{add};
  graph.SetInputs(inputs).SetOutputs(outputs);
  return true;
}
```

关键约束：

| 约束 | 说明 |
| --- | --- |
| 算子名唯一 | 同一图中算子 name 不能重复 |
| 多输入顺序对齐 | `SetInputs` 的入参顺序必须与 Data 的 `index` 属性顺序一致，否则编译报错 |
| Data 必须设描述 | Data 算子必须 `update_input_desc_x` / `update_output_desc_y` 设 shape/format/dtype |

> 维度补齐规则：构图时如果 Tensor 的 shape 维数和 format 维数不一致（例如 shape 是 1 维 `[16]`、format 是 4 维 NHWC），GE 会按既定规则补维（此例理解为 C 轴，补成 `[1,1,1,16]`）。一般建议直接用 ND 格式、shape 维数与 format 对齐，避免歧义。

## 6. ES 极简构图（C 风格 / C++ / Python）

**ES（Eager Style）** 是一套**函数风格**的极简构图 API，语法类似 Torch Eager 脚本：每个 IR 算子映射为一个函数，函数名取自算子类型，参数依次对应"输入 + 属性"，返回值对应输出。相比算子原型风格，ES 代码量更少，并提供 **C 风格、C++ 和 Python** 三种接口形式。

> 编译说明：C 风格 ES 公共头当前包含 `<cstdint>` 等 C++ 内容，需要按 C++ 源文件编译。下文的“C 风格”表示接口形式，不代表可以直接使用标准 C 编译器。

### 6.1 ES API 与 IR 的映射

以算子 `Foo`（输入 x1、x2，属性 a1、a2，输出 y1）为例：

| 接口形式 | 函数原型 |
| --- | --- |
| C 风格 | `EsCTensorHolder* EsFoo(EsCTensorHolder* x1, EsCTensorHolder* x2, int64_t a1, int64_t a2);` |
| C++ | `EsTensorHolder Foo(const EsTensorLike &x1, const EsTensorLike &x2, int64_t a1 = 10, int64_t a2 = 20);` |
| Python | `def Foo(x1, x2, *, a1: int = 10, a2: int = 20) -> TensorHolder:` |

可见 C 风格接口的函数名带 `Es` 前缀；C++/Python 中输入用位置参数、属性用默认参数/关键字参数。

### 6.2 ES 构图四步

```
创建图构建器 → 添加起始节点(输入/常量) → 添加中间节点(计算) → 设置图输出
```

完成上述四步后，再调用 `BuildAndReset` / `build_and_reset` 结束本次构图并取出最终 `Graph`。构图过程涉及两个对象：`EsGraphBuilder`（构图辅助类，只在构图阶段存在）和 `Graph`（最终产物）。

### 6.3 三种接口形式构建 Add 图

**C++**（加减乘除等运算符被重载，可直接用 `+`）：

```cpp
#include <memory>
#include "es_math_ops.h"

namespace ge {
namespace es {

std::unique_ptr<ge::Graph> BuildAddGraph() {
  EsGraphBuilder builder("graph_name");
  auto inputs = builder.CreateInputs<2>();
  EsTensorHolder add = inputs[0] + inputs[1];  // 运算符重载表达 Add
  (void)builder.SetOutput(add, 0);
  return builder.BuildAndReset();
}

}  // namespace es
}  // namespace ge
```

**Python**：

```python
from ge.es import GraphBuilder

builder = GraphBuilder("graph_name")
data0, data1 = builder.create_inputs(2)
add = data0 + data1                 # 运算符重载，等价 Add(data0, data1)
builder.set_graph_output(add, 0)
graph = builder.build_and_reset()
```

**C 风格接口**（当前需按 C++ 源文件编译）：

```cpp
#include <memory>

#include "graph/graph.h"
#include "es_Add_c.h"

std::unique_ptr<ge::Graph> BuildAddGraphWithCStyleApi() {
  auto builder =
      std::unique_ptr<EsCGraphBuilder, void (*)(EsCGraphBuilder *)>(
          EsCreateGraphBuilder("graph_name"), EsDestroyGraphBuilder);
  if (builder == nullptr) {
    return nullptr;
  }

  EsCTensorHolder *data0 = EsCreateGraphInput(builder.get(), 0);
  EsCTensorHolder *data1 = EsCreateGraphInput(builder.get(), 1);
  if (data0 == nullptr || data1 == nullptr) {
    return nullptr;
  }

  EsCTensorHolder *add = EsAdd(data0, data1);  // C 风格接口带 Es 前缀
  if (add == nullptr || EsSetGraphOutput(add, 0) != 0U) {
    return nullptr;
  }

  EsCGraph *raw_graph = EsBuildGraphAndReset(builder.get());
  return std::unique_ptr<ge::Graph>(
      static_cast<ge::Graph *>(static_cast<void *>(raw_graph)));
}
```

这里使用单算子头 `es_Add_c.h`，并将构图过程放入函数体。builder 由 `EsDestroyGraphBuilder` 释放；`EsBuildGraphAndReset` 返回的 Graph 所有权交给调用方，因此示例转换成 `std::unique_ptr<ge::Graph>` 管理。

> 第一章离线推理样例 `build_add_graph()` 用的就是 Python ES：`builder.create_input(...)` 建输入、`input_x + input_y` 表达 Add、`set_graph_output(...)` 设输出、`build_and_reset()` 收尾。ES 仓还提供了控制边、动态输入/输出、可选输入、私有属性、transformer 片段等丰富样例（`ge/examples/es/`）。

### 6.4 动手实践：构建 Add 图并在 NPU 上执行

下面的单元直接调用 CANN ES API 构建两输入 Add 图，dump AscendIR 后交给 GE Session **在线编译并在 0 号 NPU 上执行**，最后把真实设备结果与 NumPy 期望值对拍。

运行前请确认已安装 CANN toolkit/ops、执行过 `source set_env.sh`，且当前用户可以访问 0 号 NPU。本例不再提供 CPU 降级路径：缺少 CANN 或 NPU 时会直接报错，避免把 Host 模拟误当成真机验证。


In [ ]:
import warnings
warnings.filterwarnings("ignore", category=SyntaxWarning)
# === 真机运行：ES 构图 -> GE 在线编译 -> NPU 执行 -> 数值对拍 ===
import os
from pathlib import Path

import numpy as np
from ge.es.graph_builder import GraphBuilder
from ge.ge_global import GeApi
from ge.graph import DumpFormat, Tensor
from ge.graph.types import DataType, Format
from ge.session import Session

DEVICE_ID = 0
GRAPH_ID = 1

# 1. 构建一张静态 Add 图。
builder = GraphBuilder("NotebookAddGraph")
x = builder.create_input(
    index=0, name="input_x", data_type=DataType.DT_FLOAT, shape=[2, 3]
)
y = builder.create_input(
    index=1, name="input_y", data_type=DataType.DT_FLOAT, shape=[2, 3]
)
builder.set_graph_output(x + y, 0)
graph = builder.build_and_reset()
print("[OK] AscendIR 图已构建：", graph)

# 2. dump 图结构，便于和实际编译输入对应。
output_dir = Path("Sources/03.02").resolve()
output_dir.mkdir(parents=True, exist_ok=True)
original_dir = Path.cwd()
try:
    os.chdir(output_dir)
    graph.dump_to_file(format=DumpFormat.kTxt, suffix="notebook_add_graph")
finally:
    os.chdir(original_dir)

dump_files = sorted(output_dir.glob("*notebook_add_graph*"))
assert dump_files, "未找到 AscendIR dump 文件"
print("[OK] 图结构已 dump 到：", dump_files[-1])

# 3. GE 在线编译并在 0 号 NPU 上执行。
input_x = np.array([[1, 2, 3], [4, 5, 6]], dtype=np.float32)
input_y = np.array([[10, 20, 30], [40, 50, 60]], dtype=np.float32)
expected = input_x + input_y

ge_api = GeApi()
ge_api.ge_initialize({
    "ge.exec.deviceId": str(DEVICE_ID),
    "ge.graphRunMode": "0",
})
session = None
try:
    session = Session()
    session.add_graph(GRAPH_ID, graph)

    inputs = [
        Tensor(input_x.reshape(-1).tolist(), None, DataType.DT_FLOAT, Format.FORMAT_ND, [2, 3]),
        Tensor(input_y.reshape(-1).tolist(), None, DataType.DT_FLOAT, Format.FORMAT_ND, [2, 3]),
    ]
    outputs = session.run_graph(GRAPH_ID, inputs)
    actual = np.asarray(outputs[0].data, dtype=np.float32)
    np.testing.assert_allclose(actual, expected, rtol=1e-6, atol=1e-6)

    print("[OK] NPU 输出与 NumPy 对拍通过：")
    print(actual)
finally:
    outputs = None
    inputs = None
    # 释放 Session 引用，由 Session 析构统一释放图资源。
    session = None
    ge_api.ge_finalize()


## 7. Parser 解析框架模型（ONNX / TensorFlow / Caffe）

如果已经有训练好的框架模型，不必手工重搭——用 **Parser 接口**把模型文件直接解析成 AscendIR `Graph`。Parser 隔离了上层框架差异，把 TensorFlow pb、ONNX、Caffe 等统一转换成 GE 图。

<p align="left"><img src="./images/parser_flow.svg" alt="Parser 解析框架模型" width="85%"></p>

| 接口 | 头文件 / 库 | 解析对象 |
| --- | --- | --- |
| `aclgrphParseONNX` | `onnx_parser.h` / `libfmk_onnx_parser.so` | ONNX 模型文件 |
| `aclgrphParseONNXFromMem` | 同上 | 加载到**内存**的 ONNX 模型 |
| `aclgrphParseTensorFlow` | `tensorflow_parser.h` / `libfmk_parser.so` | TensorFlow pb |
| `aclgrphParseCaffe` | `caffe_parser.h` / `libfmk_parser.so` | Caffe prototxt + caffemodel |

### 7.1 ONNX 解析示例

```cpp
#include "onnx_parser.h"

std::string onnxPath = "../data/onnx_test.onnx";
std::map<ge::AscendString, ge::AscendString> parser_params = {
    {ge::AscendString(ge::ir_option::INPUT_FP16_NODES), ge::AscendString("input1;input2")},
    {ge::AscendString(ge::ir_option::OUTPUT),           ge::AscendString("newIssue")}};
ge::Graph graph1;
auto onnxStatus = ge::aclgrphParseONNX(onnxPath.c_str(), parser_params, graph1);
```

### 7.2 TensorFlow / Caffe 解析示例

```cpp
// TensorFlow
ge::aclgrphParseTensorFlow("../data/tf_test.pb", parser_params, graph1);

// Caffe（需 prototxt + caffemodel 两个文件）
ge::aclgrphParseCaffe("../data/caffe_test.prototxt", "../data/caffe_test.caffemodel", parser_params, graph1);
```

> 注意：部分产品形态（如 Atlas A2/A3 系列、Atlas 350 加速卡）**不支持 Caffe 框架**。解析得到的 `Graph` 可以直接喂给编译接口 `aclgrphBuildModel`，也可以先用改图接口（`graph.AddNodeByOp` / `graph.AddDataEdge` / `graph.RemoveEdge` 等）做结构修改后再编译。

## 8. Parser 解析配置参数

Parser 带配置的重载使用 `map<AscendString, AscendString>` 类型的 `parser_params`，用于在解析阶段指定输入/输出节点、shape 等。它在 ONNX、TensorFlow 接口中是第二个参数，在 Caffe 接口中位于 `model_file`、`weights_file` 之后，是第三个参数。常用参数如下：

| 参数（`ge::ir_option::`） | 作用 |
| --- | --- |
| `INPUT_FP16_NODES` | 指定输入数据类型为 FP16 的输入节点名，如 `"input1;input2"`（分号分隔） |
| `IS_INPUT_ADJUST_HW_LAYOUT` | 配合上者，指定输入是否为 FP16 + NC1HWC0 排布，取值 `false`/`true` |
| `OUT_NODES` | 指定模型输出：通用格式为 `node_name:output_index`；Caffe 额外支持 top name，ONNX 额外支持模型已声明的 output name |
| `IS_OUTPUT_ADJUST_HW_LAYOUT` | 配合 `OUT_NODES`，指定输出是否为 FP16 + NC1HWC0 |
| `OUTPUT` | 指定转图后的计算图名称 |
| `INPUT_SHAPE` | 指定/刷新输入 shape，支持静态 shape；部分产品支持动态范围（如 `"input:1~10,3,224,224"`） |
| `INPUT_DATA_NAMES` | 指定输入节点 name 与 index 的映射顺序 |
| `ENABLE_SCOPE_FUSION_PASSES` | 指定生效的 Scope 融合规则（**仅 `aclgrphParseTensorFlow` 支持**） |

配置写法示例：

```cpp
std::map<ge::AscendString, ge::AscendString> parser_params = {
    {ge::AscendString(ge::ir_option::INPUT_SHAPE), ge::AscendString("input_0_0:16,32,208,208")},
    {ge::AscendString(ge::ir_option::OUT_NODES),   ge::AscendString("output1")}};
```

> 说明：`INPUT_SHAPE` 为可选——不设置时 Parser 直接读模型里 Data 节点的 shape；设置后以此处为准，并刷新对应 Data 节点 shape，动态范围是否可用需同时核对目标产品约束。用 `OUT_NODES` 查看任意中间算子输出时，优先使用 `node_name:output_index`；ONNX 的 output-name 格式仅适用于模型已经声明的输出。

## 9. 输入规格与构图期校验

无论哪条入口构图，输入规格（shape / format / dtype）都要在构图或解析阶段确定，否则会在编译期报错。这里汇总三类高频问题与定位思路。

### 9.1 输入规格三要素

| 要素 | 全新构图设置方式 | Parser 设置方式 |
| --- | --- | --- |
| shape | `TensorDesc(Shape(...))` / ES `create_input(shape=...)` | `INPUT_SHAPE` 或读模型自带 shape |
| format | `TensorDesc(..., FORMAT_ND, ...)` | 解析自模型，可经编译参数调整 |
| dtype | `TensorDesc(..., DT_FLOAT)` | 解析自模型，或 `INPUT_FP16_NODES` 指定 |

### 9.2 构图期常见报错与定位

| 现象 | 常见原因 | 定位 / 处理 |
| --- | --- | --- |
| 算子名重复报错 | 同一图中两个显式命名的算子使用了相同 name | 为显式命名的算子设置唯一 name，或省略 name 让 GE 自动生成 |
| 多输入顺序错乱 / 编译失败 | `SetInputs` 入参顺序与 Data 的 `index` 不一致 | 让 inputs 顺序与各 Data 的 `index` 严格对齐 |
| Data 未设描述 | 漏调 `update_input_desc_x`/`update_output_desc_y` | Data 必须设置 shape/format/dtype |
| 算子不支持 | 网络中存在 GE 未内置的算子 | 查"Ascend IR 算子规格说明"；不支持则用 Ascend C 自定义算子（见 [3.5 自定义算子入图](./03.05_custom_op_integration.ipynb)） |
| shape 与 format 维数不符 | 如 shape 1 维、format 4 维 | 按补维规则理解，或直接用 ND + 对齐维数 |

> 提示：构图后可调用 `graph.dump_to_file(...)`（Python ES）把图 dump 成 onnx/pbtxt 可视化文件，检查节点、连边、shape 是否符合预期，再进入编译。这是排查"图构错"最直接的手段。

## 10. 两种方式的选择建议

有现成框架模型就用 **Parser**；要从零搭/精细控制就用**全新构建**，其中追求代码简洁优先选 **ES**，需要逐属性精细配置或对接已有 `op::` 代码则用**算子原型**。

| 维度 | 算子原型（C++ `op::`） | ES 极简构图 | Parser 解析 |
| --- | --- | --- | --- |
| 是否需要已有模型 | 否 | 否 | **是**（onnx/pb/caffe） |
| 代码量 | 较多 | **少** | 最少（一行解析） |
| 控制粒度 | **最细**（逐输入/属性/连边） | 较细（函数式） | 受限于原模型结构 |
| 接口形式 | C++ | **C 风格 / C++ / Python**（CANN 9.1.0 的 C 风格头文件当前需按 C++ 编译） | C++ |
| 典型场景 | 全新网络、精细改图、对接已有 IR 代码 | 快速实验、原型验证、脚本化构图 | 已训练模型迁移上昇腾 |

实践组合也很常见：先用 **Parser 解析**得到 Graph，再用**改图接口**插入/删除节点做结构调整，最后统一编译。

> 无论选择哪种入口，产出的都是 AscendIR Graph，并可进入统一的 GE 编译流程。在图结构、属性和输入输出描述等价的前提下，不同入口可以表达等价的计算语义；入口不同并不保证实际生成的图和编译结果天然完全相同。下一节 3.3 我们就来看：拿到图之后，怎么通过编译配置控制它的性能与精度。

## 课后练习

本节讲了两条构图入口（算子原型 / ES / Parser）、算子定义与连边、Graph 输入输出设置、Parser 配置与构图校验。请完成以下题目自测。

1. （判断题）GE 全新构建和 Parser 解析两条入口产出的图结构不同，编译流程也各不相同。

2. （判断题）使用 `op::Data` 定义数据节点时，必须通过 `update_input_desc_x` / `update_output_desc_y` 设置 shape、format、dtype。

3. （单选题）以下哪个接口用于将 ONNX 模型文件解析为 AscendIR Graph？
    A. `aclgrphParseTensorFlow`
    B. `aclgrphParseCaffe`
    C. `aclgrphParseONNX`
    D. `aclgrphBuildModel`

4. （单选题）关于算子原型中的注册宏，以下说法正确的是？
    A. `REQUIRED_ATTR` 注册的属性有默认值，可以不设置
    B. `DYNAMIC_INPUT` 用于注册个数不固定的动态多输入
    C. `OPTIONAL_INPUT` 注册的是必选输入
    D. `OP_END_FACTORY_REG` 用于注册算子的第一个输入

5. （多选题）以下关于 C++ 算子原型构图的描述，哪些是正确的？
    A. 同一图中算子的 name 必须唯一
    B. 必选/可选输入通过 `set_input_<输入名>` 设置
    C. 控制边通过 `AddControlInput` 表达，只约束执行顺序、不传数据
    D. `SetInputs` 的入参顺序可以任意，与 Data 的 index 属性无关

6. （多选题）以下关于 ES 极简构图的描述，哪些是正确的？
    A. ES 是函数风格 API，提供 C 风格、C++ 和 Python 三种接口形式
    B. C++ 和 Python 中加减乘除等运算符被重载，可直接用 `+` 表达 Add
    C. C 风格接口的函数名带 `Es` 前缀（如 `EsAdd`）
    D. ES 构图产出的图与算子原型构图产出的图不是同一种 IR

7. （单选题）Parser 解析配置参数中，`ENABLE_SCOPE_FUSION_PASSES` 的限制是？
    A. 所有 Parser 接口都支持
    B. 仅 `aclgrphParseONNX` 支持
    C. 仅 `aclgrphParseTensorFlow` 支持
    D. 仅 `aclgrphParseCaffe` 支持

**执行以下代码获取答案。**

In [ ]:
!cat ./answer/03.02_answer.txt